# 01 — Environment Testing

This notebook proves the inventory environment is valid and behaves sensibly before any learning agent is trained:

1. `check_env` from Stable-Baselines3 validates the Gymnasium API (spaces, `reset`/`step`, observation and reward values).
2. A random agent interacts with the environment for 30–90 days.
3. The daily stock trajectory is printed so inventory dynamics can be inspected.

The environment is built from `config/environment_config.yaml` and is assumed to take the parsed config in its constructor.

In [ ]:
import sys
from pathlib import Path

import yaml
from stable_baselines3.common.env_checker import check_env

# Make the project packages importable when the notebook runs from notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from environment.inventory_env import InventoryEnv

## 1. Validate the environment

`check_env` raises an error if the environment violates the Gymnasium API, so a clean run is the proof of validity.

In [ ]:
with open(PROJECT_ROOT / "config" / "environment_config.yaml", "r") as f:
    env_config = yaml.safe_load(f)

env = InventoryEnv(**(env_config or {}))

check_env(env)
print("check_env passed: the environment is a valid Gymnasium environment.")

## 2. Random-agent rollout

Run a uniformly random policy for a horizon in the 30–90 day range and print the stock level after every step.

In [ ]:
N_DAYS = 60  # any horizon between 30 and 90 days
assert 30 <= N_DAYS <= 90

obs, info = env.reset(seed=42)
env.action_space.seed(42)

print(f"{'Day':>3}  {'Action':>8}  {'Demand':>6}  {'Stock':>6}  {'Reward':>8}")
print("-" * 42)

for day in range(1, N_DAYS + 1):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

    stock = info.get("stock_level")
    if stock is None:
        stock = env.unwrapped.inventory_manager.stock_level
    demand = info.get("demand", "-")

    print(f"{day:>3}  {str(action):>8}  {str(demand):>6}  {str(stock):>6}  {reward:>8.2f}")

    if terminated or truncated:
        print(f"Episode ended early on day {day}.")
        break

env.close()